# 02 - Pré-processamento dos Dados
## Lending Club Loan Data

**Tech Challenge - Fase 04 | Machine Learning Engineering**

Este notebook transforma os achados da análise exploratória em um conjunto de dados consistente e reproduzível para o modelo de **Credit Scoring**.

O objetivo é preparar os dados sem introduzir vazamento de informação entre treino e teste, preservando também um **Reference Dataset** adequado para as etapas posteriores de observabilidade e drift.

### Etapas

1. carregar apenas a população com desfecho conhecido;
2. aplicar transformações determinísticas e feature engineering;
3. separar features, target e metadados temporais;
4. dividir treino e teste de forma estratificada;
5. aprender imputação, scaling e encoding apenas no treino;
6. validar a transformação;
7. gerar um Reference Dataset representativo.

> **Princípio central:** estatísticas aprendidas dos dados, como medianas, escalas e categorias do One-Hot Encoding, só podem ser ajustadas **após** o train/test split. Isso evita que informações do conjunto de teste influenciem o treinamento.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

COLORS = {
    "blue": "#377DFF",
    "teal": "#00A6A6",
    "green": "#2CB67D",
    "gold": "#F4B942",
    "coral": "#FF6B6B",
    "purple": "#7B61FF",
    "gray": "#6B7280",
}

plt.rcParams.update({
    "figure.figsize": (9, 4.8),
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# 1. Caminhos e definição da população

In [ ]:
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")
PROCESSED_PATH = Path("../data/processed")
REFERENCE_PATH = Path("../data/reference")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
REFERENCE_PATH.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset não encontrado em: {DATA_PATH.resolve()}\n"
        "Coloque accepted_2007_to_2018Q4.csv em data/raw/."
    )

print(f"Dataset: {DATA_PATH.resolve()}")

In [ ]:
GOOD_STATUS = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid",
]

BAD_STATUS = [
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off",
]

BASELINE_FEATURES = [
    "loan_amnt",
    "term",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "earliest_cr_line",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "collections_12_mths_ex_med",
    "acc_now_delinq",
    "pub_rec_bankruptcies",
    "tax_liens",
    "application_type",
]

## 1.1 Leitura em chunks

A base bruta possui mais de 2 milhões de registros e 151 colunas. Para reduzir o consumo de memória, serão lidas somente as features aprovadas no EDA, além de `loan_status` e `issue_d`.

In [ ]:
cols_to_load = BASELINE_FEATURES + ["loan_status", "issue_d"]

chunks = []

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=cols_to_load,
    chunksize=100_000,
    low_memory=False,
):
    chunk = chunk[
        chunk["loan_status"].isin(GOOD_STATUS + BAD_STATUS)
    ]
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
del chunks

print(f"Registros carregados: {len(df):,}")
print(f"Colunas carregadas:   {df.shape[1]}")

In [ ]:
df["target"] = (
    df["loan_status"]
    .isin(BAD_STATUS)
    .astype(int)
)

print(df["target"].value_counts())
print()
print(df["target"].value_counts(normalize=True))

A população de modelagem contém apenas empréstimos com desfecho conhecido, mantendo aproximadamente **80% de não inadimplentes e 20% de inadimplentes**.

# 2. Feature engineering e correções determinísticas

As transformações desta seção não aprendem estatísticas da base. Por isso podem ser realizadas antes do split.

As operações incluem:

- converter o prazo do contrato para número de meses;
- transformar tempo de emprego em anos;
- preservar explicitamente a ausência de `emp_length`;
- consolidar o intervalo FICO em uma média;
- transformar datas em tempo de histórico de crédito;
- converter valores claramente inválidos em `NaN`, para posterior imputação no pipeline.

## 2.1 Prazo do empréstimo

In [ ]:
df["term"] = (
    df["term"]
    .str.extract(r"(\d+)", expand=False)
    .astype(int)
)

df["term"].value_counts().sort_index()

`term` deixa de ser texto (`36 months`, `60 months`) e passa a representar diretamente a quantidade de meses do contrato.

## 2.2 Tempo de emprego

In [ ]:
EMP_LENGTH_MAP = {
    "< 1 year": 0,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9,
    "10+ years": 10,
}

df["emp_length_years"] = df["emp_length"].map(EMP_LENGTH_MAP)

# A ausência apresentou sinal no EDA, portanto será preservada em indicador próprio.
df["emp_length_missing"] = df["emp_length"].isna().astype(int)

# -1 funciona como representação explícita de "não informado".
df["emp_length_years"] = df["emp_length_years"].fillna(-1)

df[["emp_length", "emp_length_years", "emp_length_missing"]].head()

A ausência de `emp_length` não é descartada. O EDA mostrou que esse grupo apresenta comportamento de inadimplência diferente, portanto a informação é preservada por `emp_length_missing`.

## 2.3 FICO médio

In [ ]:
df["fico_avg"] = (
    df["fico_range_low"] + df["fico_range_high"]
) / 2

df["fico_avg"].describe()

As duas extremidades do intervalo FICO são consolidadas em `fico_avg`, reduzindo redundância sem perder a informação central do score.

## 2.4 Tempo de histórico de crédito

In [ ]:
df["issue_date"] = pd.to_datetime(
    df["issue_d"],
    format="%b-%Y",
    errors="coerce"
)

df["earliest_cr_date"] = pd.to_datetime(
    df["earliest_cr_line"],
    format="%b-%Y",
    errors="coerce"
)

df["credit_history_years"] = (
    (df["issue_date"] - df["earliest_cr_date"]).dt.days / 365.25
)

df["credit_history_years"].describe()

In [ ]:
negative_credit_history = (df["credit_history_years"] < 0).sum()

print(f"Históricos de crédito negativos: {negative_credit_history:,}")

# Proteção contra eventual inconsistência temporal.
df.loc[
    df["credit_history_years"] < 0,
    "credit_history_years"
] = np.nan

Os valores encontrados são plausíveis e não indicam inconsistência temporal relevante. A proteção acima mantém o pipeline robusto caso novos dados apresentem datas invertidas.

## 2.5 Tratamento de DTI inválido

In [ ]:
invalid_dti = (df["dti"] < 0).sum()

print(f"Registros com DTI negativo antes do tratamento: {invalid_dti:,}")

# DTI negativo não possui interpretação válida para este contexto.
# Ele é convertido para NaN e será imputado posteriormente usando somente o treino.
df.loc[df["dti"] < 0, "dti"] = np.nan

print(f"Registros com DTI negativo após o tratamento: {(df['dti'] < 0).sum():,}")

Valores extremos positivos de DTI **não são removidos automaticamente**. O EDA mostrou caudas longas e o projeto prioriza distinguir observações raras de valores realmente inválidos.

# 3. Validação das features derivadas

In [ ]:
derived_summary = df[
    [
        "term",
        "emp_length_years",
        "emp_length_missing",
        "fico_avg",
        "credit_history_years",
        "dti",
    ]
].describe().T

derived_summary

In [ ]:
derived_missing = pd.DataFrame({
    "missing_count": df[
        [
            "term",
            "emp_length_years",
            "emp_length_missing",
            "fico_avg",
            "credit_history_years",
            "dti",
        ]
    ].isna().sum()
})

derived_missing["missing_percent"] = (
    derived_missing["missing_count"] / len(df) * 100
)

derived_missing

### Validação

- `term` permanece restrito aos prazos esperados de 36 e 60 meses;
- `emp_length_years` preserva ausência como `-1`, acompanhada de indicador próprio;
- `fico_avg` substitui o par original de FICO;
- `credit_history_years` apresenta valores positivos e plausíveis;
- DTI negativo é convertido para ausente e será imputado somente após o split.

# 4. Separação de features, target e metadados

`issue_date` é preservada como **metadado temporal** para observabilidade e drift, mas não entra inicialmente como feature preditora.

In [ ]:
metadata = df[["issue_date", "target"]].copy()

X = df.drop(
    columns=[
        "loan_status",
        "target",
        "emp_length",
        "fico_range_low",
        "fico_range_high",
        "earliest_cr_line",
        "issue_d",
        "issue_date",
        "earliest_cr_date",
    ]
)

y = df["target"].copy()

print(f"X: {X.shape}")
print(f"y: {y.shape}")

In [ ]:
X.head()

# 5. Train/Test Split

A divisão é realizada **antes** de imputação, escalonamento e encoding. Dessa forma, o conjunto de teste permanece completamente fora do aprendizado das transformações.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Treino: {X_train.shape}")
print(f"Teste:  {X_test.shape}")

In [ ]:
split_distribution = pd.DataFrame({
    "Treino": y_train.value_counts(normalize=True).sort_index(),
    "Teste": y_test.value_counts(normalize=True).sort_index(),
}).T

split_distribution.columns = ["Não inadimplente", "Inadimplente"]
split_distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

x = np.arange(len(split_distribution.index))
width = 0.32

ax.bar(
    x - width/2,
    split_distribution["Não inadimplente"] * 100,
    width,
    label="Não inadimplente",
    color=COLORS["teal"]
)

ax.bar(
    x + width/2,
    split_distribution["Inadimplente"] * 100,
    width,
    label="Inadimplente",
    color=COLORS["coral"]
)

ax.set_xticks(x)
ax.set_xticklabels(split_distribution.index)
ax.set_ylabel("Percentual")
ax.set_title("Distribuição do target após o split estratificado")
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

O `stratify=y` preserva praticamente a mesma distribuição de classes nos conjuntos de treino e teste.

# 6. Pipeline de pré-processamento

O pipeline utiliza tratamentos diferentes para features numéricas e categóricas.

### Numéricas
- imputação pela **mediana**, calculada somente no treino;
- `RobustScaler`, menos sensível às caudas e outliers identificados no EDA.

### Categóricas
- categorias ausentes recebem `Unknown`;
- `OneHotEncoder(handle_unknown="ignore")` permite processar categorias não vistas anteriormente sem derrubar o pipeline.

In [ ]:
numeric_features = (
    X_train
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(exclude=["number"])
    .columns
    .tolist()
)

print(f"Features numéricas: {len(numeric_features)}")
print(numeric_features)

print(f"\nFeatures categóricas: {len(categorical_features)}")
print(categorical_features)

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            RobustScaler()
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        ),
    ]
)

## 6.1 Ajuste somente no treino

In [ ]:
preprocessor.fit(X_train)
print("Preprocessor ajustado somente com X_train.")

## 6.2 Validação em amostra

Antes de aplicar o pipeline ao conjunto completo, a transformação é validada em 1.000 registros.

In [ ]:
X_sample = X_train.head(1_000)
X_sample_transformed = preprocessor.transform(X_sample)

print(f"Shape original:    {X_sample.shape}")
print(f"Shape transformado:{X_sample_transformed.shape}")

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print(
    f"Quantidade de features após transformação: "
    f"{len(feature_names)}"
)

pd.Series(feature_names, name="feature").to_frame()

In [ ]:
X_sample_array = (
    X_sample_transformed.toarray()
    if hasattr(X_sample_transformed, "toarray")
    else X_sample_transformed
)

validation_results = pd.Series({
    "NaNs após preprocessing": int(np.isnan(X_sample_array).sum()),
    "Infinitos após preprocessing": int(np.isinf(X_sample_array).sum()),
})

validation_results.to_frame("count")

O pipeline transforma as 23 features originais em aproximadamente **44 features**, devido principalmente ao One-Hot Encoding. A validação deve retornar **zero NaNs e zero infinitos**.

# 7. Reference Dataset para observabilidade

O modelo será treinado com todo o conjunto de treino. Para observabilidade, porém, um Reference Dataset menor torna cálculos como PSI, KS e relatórios do Evidently mais leves.

Será criada uma referência estratificada com **100.000 registros**, preservando:

- as features em formato interpretável, antes do scaling/One-Hot;
- a variável alvo;
- `issue_date` como metadado temporal.

> O Reference Dataset não substitui o conjunto de treinamento.

In [ ]:
reference_idx, _ = train_test_split(
    X_train.index,
    train_size=100_000,
    random_state=42,
    stratify=y_train,
)

reference_df = X_train.loc[reference_idx].copy()

reference_df["target"] = y_train.loc[
    reference_idx
].values

reference_df["issue_date"] = metadata.loc[
    reference_idx,
    "issue_date"
].values

print(f"Shape do Reference Dataset: {reference_df.shape}")
reference_df["target"].value_counts(normalize=True)

In [ ]:
reference_distribution = (
    reference_df["target"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

fig, ax = plt.subplots(figsize=(7, 4.2))

bars = ax.bar(
    ["Não inadimplente", "Inadimplente"],
    reference_distribution.values,
    color=[COLORS["teal"], COLORS["coral"]]
)

ax.set_title("Distribuição do target no Reference Dataset")
ax.set_ylabel("Percentual")
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.set_ylim(0, reference_distribution.max() * 1.18)

for bar, value in zip(bars, reference_distribution.values):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 1,
        f"{value:.2f}%",
        ha="center",
        fontweight="bold"
    )

plt.tight_layout()
plt.show()

## 7.1 Persistência local da referência

A pasta `data/` permanece no `.gitignore`, portanto o arquivo é criado localmente para uso pelas próximas etapas sem ser versionado no repositório.

In [ ]:
reference_file = REFERENCE_PATH / "reference_dataset.csv"

reference_df.to_csv(
    reference_file,
    index=False
)

print(f"Reference salvo em: {reference_file}")
print(f"Arquivo existe: {reference_file.exists()}")
print(f"Shape: {reference_df.shape}")

# 8. Checklist de validação do pré-processamento

In [ ]:
checks = {
    "target_binario": set(y.unique()).issubset({0, 1}),
    "term_valido": set(X["term"].dropna().unique()).issubset({36, 60}),
    "historico_credito_sem_negativos": not (X["credit_history_years"].dropna() < 0).any(),
    "dti_sem_negativos": not (X["dti"].dropna() < 0).any(),
    "split_estratificado": abs(y_train.mean() - y_test.mean()) < 0.001,
    "preprocessor_sem_nan": np.isnan(X_sample_array).sum() == 0,
    "preprocessor_sem_inf": np.isinf(X_sample_array).sum() == 0,
    "reference_100k": len(reference_df) == 100_000,
    "reference_tem_issue_date": "issue_date" in reference_df.columns,
}

validation_table = (
    pd.Series(checks, name="aprovado")
    .to_frame()
)

validation_table

In [ ]:
if not validation_table["aprovado"].all():
    failed = validation_table.index[
        ~validation_table["aprovado"]
    ].tolist()
    raise AssertionError(
        f"Falharam as validações: {failed}"
    )

print("Todas as validações do pré-processamento foram aprovadas.")

# 9. Conclusões do pré-processamento

O pré-processamento foi estruturado para manter separadas as transformações determinísticas e as transformações que precisam aprender parâmetros dos dados.

### Transformações realizadas
- `term` convertido para meses;
- `emp_length` convertido para anos, preservando ausência em indicador próprio;
- `fico_avg` criado a partir do intervalo FICO;
- `credit_history_years` criado a partir das datas de primeira linha de crédito e concessão;
- valores negativos de DTI tratados como ausentes;
- features obsoletas após feature engineering removidas do conjunto preditivo.

### Proteção contra Data Leakage
O train/test split ocorre antes da imputação, scaling e One-Hot Encoding. O `ColumnTransformer` é ajustado exclusivamente sobre `X_train`.

### Pipeline
- numéricas: `SimpleImputer(strategy="median")` + `RobustScaler`;
- categóricas: imputação com `Unknown` + `OneHotEncoder(handle_unknown="ignore")`.

### Reference Dataset
Foi criado um conjunto de referência estratificado com 100.000 registros, contendo target e metadado temporal. Ele será utilizado nas etapas posteriores de monitoramento e drift.

### Próxima etapa
A lógica validada neste notebook será migrada para `src/credit_scoring_observability/preprocessing.py`, permitindo que treinamento, inferência e monitoramento utilizem exatamente as mesmas transformações.

---
### Nota de reprodutibilidade

Antes do commit, execute **Run All** com o kernel do ambiente `.venv` e confirme que a última validação retorna:

`Todas as validações do pré-processamento foram aprovadas.`

O dataset bruto e os arquivos gerados em `data/` permanecem fora do versionamento Git.